# 3a — Mapbox Traffic

Fetches Mapbox traffic + streets MVT tiles, spatially joins them, and map-matches
the result to OSM driving edges, writing congestion to `mapbox_congestion_history`.

Requires `MAPBOX_ACCESS_TOKEN` in `../.env`.

---

## How it works

Mapbox exposes two tile endpoints at the same zoom/x/y coordinates:

| Tileset | Content | Layer |
|---|---|---|
| `mapbox.mapbox-traffic-v1` | Congestion levels per road segment | `traffic` |
| `mapbox.mapbox-streets-v8` | Road geometry with OSM attributes | `road` |

Both tilesets are downloaded in parallel as MVT (Mapbox Vector Tiles) binary files.
The traffic tile carries a `congestion` property (`low` / `moderate` / `heavy` / `severe`).

**Step 1 — Tile fetch:** download both tilesets for every tile intersecting the boundary.
Already-downloaded `.mvt` files are reused from disk.

**Step 2 — Spatial join:** decode MVT → WGS-84, then `sjoin_nearest` (max 20 m) to attach
each traffic segment's congestion to the nearest streets-v8 road.

**Step 3 — Map match:** reproject to EPSG:3857 and match Mapbox segments to OSM edges
using a 25 m corridor + bearing filter (≤ 45°) + 40% overlap threshold.
Most severe congestion wins when multiple segments compete for the same edge.

**Note:** Mapbox assigns `low` to all free-flowing roads, so coverage is near 100%.
Google, by contrast, only marks congested roads — uncongested roads show as `no data`.

In [ ]:
%pip install mercantile mapbox-vector-tile python-dotenv folium --quiet

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

sys.path.insert(0, '..')
from scripts.mapbox_traffic import MapboxTraffic
from scripts.traffic_db import TrafficDB, CONGESTION_COLORS

load_dotenv(Path('../.env'), override=True)

# ── Configuration ─────────────────────────────────────────────────────
NAME      = 'sodermalm'
ZOOM      = 14
DB        = f'../db/{NAME}.duckdb'
BOUNDARY  = f'../boundaries/{NAME}.geojson'
TILES_DIR = Path('../tiles')
OUT_FILE  = Path(f'../output/{NAME}_traffic_mapbox.geojson')
TOKEN     = os.environ.get('MAPBOX_ACCESS_TOKEN', '')
# ─────────────────────────────────────────────────────────────────────

if not TOKEN:
    raise ValueError('MAPBOX_ACCESS_TOKEN not set in ../.env')
print(f'Token    : {TOKEN[:8]}...{TOKEN[-4:]}')
print(f'DuckDB   : {DB}')
print(f'Boundary : {BOUNDARY}')
print(f'Zoom     : {ZOOM}  (~{40_075_016 // (512 * 2**ZOOM)} m/tile side)')

---
## Step 1 — Fetch tiles

Downloads Mapbox traffic-v1 and streets-v8 MVT tiles for every tile that intersects
the boundary, then spatially joins them and clips to the boundary.

Tiles already on disk are **reused without re-downloading**.
The output GeoJSON is also cached — delete `OUT_FILE` to force a fresh fetch.

Congestion values: `low` · `moderate` · `heavy` · `severe` · `no data`

In [ ]:
%%time
traffic = MapboxTraffic(token=TOKEN, zoom=ZOOM)
gdf = traffic.fetch(BOUNDARY, TILES_DIR, OUT_FILE)

print(f'Segments   : {len(gdf):,}')
print(f'Congestion : {gdf["congestion"].value_counts().to_dict()}')
display(gdf.head(3))

---
## Step 2 — Map match to OSM edges

Matches each congested Mapbox segment to OSM `driving.edges`:

1. Project to EPSG:3857 (metres)
2. Buffer each traffic segment by **25 m** to create a corridor
3. Keep OSM edges inside the corridor where:
   - **Bearing difference ≤ 45°** (edge and segment point roughly the same direction)
   - **≥ 40% of the OSM edge** lies inside the corridor
4. **Most severe congestion wins** when multiple segments match one edge

Expected match rate: **95–99%** for Mapbox (nearly all roads get at least `low`).

In [ ]:
%%time
edge_cong = traffic.map_match(DB, gdf)

from collections import Counter
print(f'Edges matched : {len(edge_cong):,}')
print(f'Distribution  : {Counter(edge_cong.values())}')

---
## Step 3 — Write to DuckDB

Appends one row per matched edge to `mapbox_congestion_history` and one row to `runs`.
Re-running creates a new `run_id` — the history is append-only.

In [ ]:
%%time
with TrafficDB(DB, read_only=False) as db:
    run_id = db.write_congestion(
        edge_cong,
        source        = 'mapbox',
        zoom          = ZOOM,
        n_segments    = len(gdf),
        boundary_name = NAME,
    )
print(f'Written as run_id={run_id}  source=mapbox  zoom={ZOOM}')

---
## Step 4 — Inspect results

Typical Mapbox distribution: ~97% `low`, ~1-2% `moderate`/`heavy`/`severe`, near-zero `no data`.

In [ ]:
%%time
with TrafficDB(DB) as db:
    print('=== History ===')
    display(db.get_history_index())
    print('\n=== Congestion summary (Mapbox) ===')
    display(db.get_congestion_summary('mapbox'))

    edges = db.get_edges(source='mapbox')
    m = db.plot_edges(edges)
m